# VLLM

## Embedding

In [ ]:
## Using vLLM offline
from vllm import LLM, PoolingParams
import numpy as np
model = LLM(
    model = "./models/Qwen/Qwen3-Embedding-0.6B",
    task = "embed",
    enforce_eager=True,
)

outputs = model.embed(["我爱你", "我喜欢你"], pooling_params=PoolingParams(normalize=True))
embeds = []
for output in outputs:
    embed = output.outputs.embedding
    embed = np.array(embed).astype(np.float32)
    print(embed)
    embeds.append(embed)
print(embeds[0].dot(embeds[1]))

In [ ]:
## Using OpenAI API online
from openai import OpenAI
import numpy as np
openai_api_key = "EMPTY"
openai_base_url = "http://localhost:8081/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_base_url
)

models = client.models.list()
model = models.data[0].id
print("Using model:", model)

response = client.embeddings.create(
    input = ["我爱你", "我喜欢你"],
    model=model,
    normalize=True,
)
embedding1 = np.array(response.data[0].embedding).astype(np.float32)
embedding2 = np.array(response.data[1].embedding).astype(np.float32)

print(embedding1.dot(embedding2))

In [ ]:
## 使用AsyncLLM异步接口
from vllm.v1.engine.async_llm import AsyncLLM
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.pooling_params import PoolingParams

engine_args = AsyncEngineArgs(
    model="./models/Qwen/Qwen3-Embedding-0.6B",
    task="embed",
    enforce_eager=True,
)

engine = AsyncLLM.from_engine_args(engine_args)
pooling_params = PoolingParams(normalize=True, task="embed")
import time
async def async_embed(text):
    response = engine.encode(text, pooling_params=pooling_params, request_id=str(time.time()))
    final_output = None
    async for output in response:
        final_output = output
        print(output)
    if final_output and final_output.outputs:
        return final_output.outputs
    return None

import asyncio
async def main():
    texts = ["要不要来一瓶可乐", "我喜欢你"]
    tasks = [async_embed(text) for text in texts]
    results = await asyncio.gather(*tasks)
    for i, embeds in enumerate(results):
        print(f"Text: {texts[i]}")
        print(f"Embedding: {embeds}")
    print(results[0].data.numpy() @ results[1].data.numpy().T)

await main()

## Rerank

In [ ]:
## 使用vLLM进行Rerank
def format_query(query):
    prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
    instruction = 'Given a web search query, retrieve relevant passages that answer the query'
    return prefix + "<Instruct>: {instruction}\n<Query>: {query}\n".format(instruction=instruction,query=query)

def format_doc(doc):
    suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    return "<Document>: " + doc + suffix


query = "我爱你"
docs = ["我喜欢喝可乐", "我喜欢你"]

from vllm import LLM
model = LLM(
    model="./models/Qwen/Qwen3-Reranker-4B",
    task="score",
    hf_overrides={
        "architectures": ["Qwen3ForSequenceClassification"],
        "classifier_from_token": ["no", "yes"],
        "is_original_qwen3_reranker": True
    },
    enforce_eager=True,
)

responses = model.score(format_query(query), [format_doc(doc) for doc in docs])

for response in responses:
    print(f"Score: {response.outputs.score}")

In [6]:
## 使用requests
import requests
model_name = "models/Qwen/Qwen3-Reranker-4B"

# score
url = "http://localhost:8082/score"
response = requests.post(
    url=url,
    json={
        "model": model_name,
        "text_1": format_query(query),
        "text_2": [format_doc(doc) for doc in docs]
    }
).json()

print(response)

# rerank
url = "http://localhost:8082/v1/rerank"
response = requests.post(
    url=url,
    json={
        "model": model_name,
        "query": format_query(query),
        "documents": [format_doc(doc) for doc in docs],
        "top_n": 2
    }
).json()
print(response)

{'id': 'score-798d624deae1435facf0937a61fcbd21', 'object': 'list', 'created': 1760926579, 'model': 'models/Qwen/Qwen3-Reranker-4B', 'data': [{'index': 0, 'object': 'score', 'score': 0.07909675687551498}, {'index': 1, 'object': 'score', 'score': 0.969491183757782}], 'usage': {'prompt_tokens': 160, 'total_tokens': 160, 'completion_tokens': 0, 'prompt_tokens_details': None}}
{'id': 'rerank-36d49d52db674e3da444dfe47b591d4d', 'model': 'models/Qwen/Qwen3-Reranker-4B', 'usage': {'total_tokens': 160}, 'results': [{'index': 1, 'document': {'text': '<Document>: 我喜欢你<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n', 'multi_modal': None}, 'relevance_score': 0.969491183757782}, {'index': 0, 'document': {'text': '<Document>: 我喜欢喝可乐<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n', 'multi_modal': None}, 'relevance_score': 0.07909675687551498}]}


## Generate

In [ ]:
## 使用vLLM进行Generate 
from vllm import LLM, SamplingParams
model = LLM(
    model="models/Qwen/Qwen3-4B",
    enforce_eager=True,
    reasoning_parser="deepseek_r1"
)
responses = model.generate("Hello, my name is", SamplingParams(temperature=0))
for output in responses:
    print(output)
responses = model.chat(
    [
        {"role": "system", "content": "你是一个全能的人工智能助手"},
        {"role": "user", "content": "你的名字是什么"}
    ],
    sampling_params=SamplingParams(temperature=0, max_tokens=1024),
)
for response in responses:
    print(response.prompt)
    print(response.outputs[0].text)

In [ ]:
# 使用requests进行流式输出
import json
import requests

url = "http://localhost:8083/v1/chat/completions"
model = "models/Qwen/Qwen3-4B"
with requests.post(
    url=url,
    json={
        "messages": [
            {"role": "system", "content": "你是一个全能的人工智能助手"},
            {"role": "user", "content": "你的名字是什么"}
        ],
        "model": model,
        "include_reasoning": True,
        "stream": True
    },
    stream=True
) as response:

    for line in response.iter_lines():
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                if line == "[DONE]":
                     break
                line = json.loads(line)
                print(line["choices"][0]["delta"].get("reasoning_content", ""), end="")
                print(line["choices"][0]["delta"].get("content", ""), end="")

## 异步性能对比

In [ ]:
## 使用 AsyncLLM
import asyncio
import time
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
from vllm.sampling_params import SamplingParams

engine_args = AsyncEngineArgs(
    model="./models/Qwen/Qwen3-4B",
    hf_overrides={
        "trust_remote_code": True
    },
    max_model_len=1024,
    enforce_eager=True
)

engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(temperature=0, max_tokens=512)


async def async_chat(prompt):
    response =  engine.generate(prompt, sampling_params=sampling_params, request_id=str(time.time()))
    final_output = ""
    async for output in response:
        final_output = output
    if final_output and final_output.outputs:
        return final_output.outputs[0].text
    return ""

async def main():
    start = time.time()
    
    # 并发执行多个协程
    tasks = [
        async_chat("你是谁？"),
        async_chat("今天天气怎么样？"),
        async_chat("请介绍一下人工智能的发展历史。"),
        async_chat("什么是量子计算？"),
        async_chat("请解释一下区块链技术的基本原理。"),
        async_chat("机器学习和深度学习有什么区别？"),
    ]
    
    results = await asyncio.gather(*tasks)  # 等待所有任务完成
    print("结果:", results)
    print(f"总耗时: {time.time() - start:.2f} 秒")

await main()

In [ ]:
## 使用传统的transformers接口
import asyncio
import time
from modelscope import AutoModelForCausalLM, AutoTokenizer
model_name = "./models/Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, device_map="cuda")

async def chat_with_modelscope(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):]  # 返回生成的部分

async def main_modelscope():
    start = time.time()
    
    # 并发执行多个协程
    tasks = [
        chat_with_modelscope("你是谁？"),
        chat_with_modelscope("今天天气怎么样？"),
        chat_with_modelscope("请介绍一下人工智能的发展历史。"),
        chat_with_modelscope("什么是量子计算？"),
        chat_with_modelscope("请解释一下区块链技术的基本原理。"),
        chat_with_modelscope("机器学习和深度学习有什么区别？"),
    ]
    
    results = await asyncio.gather(*tasks)  # 等待所有任务完成
    print("结果:", results)
    print(f"总耗时: {time.time() - start:.2f} 秒")

await main_modelscope()